# 🗂️ Notebook 2: Airbnb — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🧩 Entities

The smallest set of things we need to model:

- **User** — can be a host, guest, or both.
- **Listing** — a property owned by a host.
- **Booking** — a confirmed date range by a guest.
- **Availability** — per-day state for each listing (*free / blocked / booked*).

That last one seems redundant — we already know bookings, right? Read on.

## 🐌 Attempt 1 (bad): bookings as ranges, no availability table

The obvious first design: store `check_in` and `check_out` on each booking. To check if dates are free, **scan all bookings for that listing** and look for overlaps.

Problems:

1. **Slow** — every booking check does a range scan.
2. **Race-prone** — two concurrent checks can both see "free" and both insert.
3. **Hard to index** — range overlap queries (`a.start < b.end AND b.start < a.end`) don't use B-trees well.
4. **Hard to show a calendar** — rendering "which days are free in May?" means walking every booking.

In [ ]:
from datetime import date, timedelta

bookings = []  # (listing_id, check_in, check_out)

def overlaps(a_in, a_out, b_in, b_out):
    # half-open ranges: [in, out)
    return a_in < b_out and b_in < a_out

def book_bad(listing_id, check_in, check_out):
    for lid, ci, co in bookings:
        if lid == listing_id and overlaps(ci, co, check_in, check_out):
            raise ValueError("dates not available")
    bookings.append((listing_id, check_in, check_out))
    return "confirmed"

print(book_bad(1, date(2026,5,1), date(2026,5,4)))
try:
    book_bad(1, date(2026,5,3), date(2026,5,6))
except ValueError as e:
    print("rejected:", e)

## ✅ Attempt 2 (better): materialize per-day availability

Instead of storing ranges, store **one row per listing-per-day** in an `availability` table:

```sql
CREATE TABLE availability (
    listing_id   BIGINT,
    day          DATE,
    booking_id   BIGINT NULL,
    PRIMARY KEY (listing_id, day)         -- <<< prevents double-booking
);
```

To book May 1–4, we insert 3 rows (nights, not days-of-stay) with the booking_id. If any row is already taken, the **primary key** causes the insert to fail atomically — the database does the hard work for us.

Below we demo this with **SQLite** — no server required.

In [ ]:
import sqlite3
from datetime import date, timedelta

con = sqlite3.connect(":memory:")
con.executescript('''
    CREATE TABLE listing (
        id INTEGER PRIMARY KEY,
        title TEXT,
        price_cents INTEGER
    );
    CREATE TABLE booking (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        listing_id INTEGER,
        guest_id INTEGER,
        check_in TEXT,
        check_out TEXT,
        status TEXT
    );
    CREATE TABLE availability (
        listing_id INTEGER,
        day TEXT,
        booking_id INTEGER,
        PRIMARY KEY (listing_id, day)
    );
''')
con.execute("INSERT INTO listing VALUES (1, 'Cozy cabin', 12000)")
con.commit()

def nights(ci: date, co: date):
    d = ci
    while d < co:
        yield d
        d += timedelta(days=1)

def book(listing_id: int, guest_id: int, ci: date, co: date) -> int:
    cur = con.cursor()
    try:
        cur.execute("BEGIN IMMEDIATE")
        cur.execute(
            "INSERT INTO booking(listing_id,guest_id,check_in,check_out,status)"
            " VALUES (?,?,?,?,'confirmed')",
            (listing_id, guest_id, ci.isoformat(), co.isoformat()),
        )
        bid = cur.lastrowid
        # Insert one availability row per night.
        # If any (listing_id, day) already exists, PK violation -> rollback.
        for d in nights(ci, co):
            cur.execute(
                "INSERT INTO availability(listing_id,day,booking_id) VALUES (?,?,?)",
                (listing_id, d.isoformat(), bid),
            )
        con.commit()
        return bid
    except sqlite3.IntegrityError:
        con.rollback()
        raise ValueError("dates not available")

print("first  :", book(1, 7, date(2026,5,1), date(2026,5,4)))
try:
    book(1, 8, date(2026,5,3), date(2026,5,6))
except ValueError as e:
    print("second : rejected ->", e)

### Why this is nicer

- **Correctness is the DB's job** — no clever code needed.
- **Calendar queries are trivial** — `SELECT day FROM availability WHERE listing_id = ? AND day BETWEEN ? AND ?`.
- **Indexes are easy** — it's just a composite PK.

⚖️ **Trade-offs, honestly:**

| Cost | Detail |
|---|---|
| More rows per booking | A 3-night stay is 4 writes (1 booking + 3 nights) instead of 1. At ~60 bookings/sec that is still trivial. |
| Long stays are worse | A 90-night sublet writes 90 rows in one transaction. Cap the max stay (Airbnb caps at 28 nights for most listings) or fall back to a range type. |
| Cancellation is a range delete | `DELETE FROM availability WHERE booking_id = ?` — fine, but it must be in the same transaction as the booking status change. |
| Only models *sold* inventory | Host-blocked dates and per-day pricing need their own table (see notebook 1). |

> Contrast with the alternative: Postgres has a native `daterange` type and an
> `EXCLUDE USING gist (listing_id WITH =, stay WITH &&)` constraint that rejects
> overlapping ranges directly — one row per booking, no per-night explosion. It is
> the better answer *if you are on Postgres*. The per-day model is the portable one,
> and it is what you can defend on a whiteboard in 30 seconds.

> Real-world Airbnb uses a similar per-day availability model internally. The "unique constraint protects correctness" pattern is one of the most useful tricks in backend engineering.

## 🧰 Typed models with Pydantic

Before exposing an HTTP API, nail down the **shapes**. Pydantic makes invalid data unrepresentable.

In [ ]:
from datetime import date
from decimal import Decimal
from typing import Literal
from pydantic import BaseModel, Field, field_validator

class Listing(BaseModel):
    id: int
    host_id: int
    title: str = Field(min_length=3, max_length=120)
    lat: float = Field(ge=-90, le=90)
    lng: float = Field(ge=-180, le=180)
    price_per_night: Decimal = Field(gt=0)
    max_guests: int = Field(ge=1, le=16)

class BookingRequest(BaseModel):
    listing_id: int
    guest_id: int
    check_in: date
    check_out: date
    guests: int = Field(ge=1)

    @field_validator("check_out")
    @classmethod
    def _after_checkin(cls, v, info):
        ci = info.data.get("check_in")
        if ci and v <= ci:
            raise ValueError("check_out must be after check_in")
        return v

class Booking(BaseModel):
    id: int
    listing_id: int
    guest_id: int
    check_in: date
    check_out: date
    status: Literal["pending", "confirmed", "cancelled"] = "pending"

    def nights(self) -> int:
        return (self.check_out - self.check_in).days

req = BookingRequest(listing_id=1, guest_id=7,
                    check_in=date(2026,5,1),
                    check_out=date(2026,5,4), guests=2)
print("valid:", req)

from pydantic import ValidationError
try:
    BookingRequest(listing_id=1, guest_id=7,
                   check_in=date(2026,5,4),
                   check_out=date(2026,5,1), guests=2)
except ValidationError as e:
    print("rejected bad request:", e.errors()[0]["msg"])

## 🌐 HTTP API sketch

A minimal, REST-ish surface. Real Airbnb has hundreds of endpoints — these are the load-bearing four.

| Method | Path | Purpose |
|---|---|---|
| `GET` | `/search?lat=..&lng=..&check_in=..&check_out=..&guests=..` | Geo + date search |
| `GET` | `/listings/{id}` | Listing detail + available dates |
| `POST` | `/bookings` | Create booking (server enforces availability) |
| `DELETE` | `/bookings/{id}` | Cancel (frees availability rows) |

### Example responses

```json
// GET /listings/1
{
  "id": 1,
  "title": "Cozy cabin",
  "price_per_night": "120.00",
  "max_guests": 4,
  "unavailable_days": ["2026-05-01", "2026-05-02", "2026-05-03"]
}
```

```json
// POST /bookings
// 201 Created
{ "id": 1001, "status": "confirmed", "total_cents": 36000 }
// 409 Conflict
{ "error": "dates not available" }
```

### Idempotency

Mobile clients retry on flaky networks. A retried `POST /bookings` **must not** create two bookings. Accept an `Idempotency-Key` header; store `(key, response)` for 24h; on replay, return the stored response.

In [ ]:
# Tiny idempotency cache.
idem_cache: dict[str, dict] = {}

def create_booking(key: str, payload: dict):
    if key in idem_cache:
        return {"replayed": True, **idem_cache[key]}
    result = {"id": len(idem_cache) + 1001, "status": "confirmed"}
    idem_cache[key] = result
    return {"replayed": False, **result}

print(create_booking("abc-123", {"listing_id": 1}))
print(create_booking("abc-123", {"listing_id": 1}))   # client retry
print(create_booking("xyz-999", {"listing_id": 1}))

## 🧠 Takeaways

- Pick a schema where **correctness invariants are enforced by the database**, not by app code.
- Typed request models catch half the bugs for free.
- Add **idempotency keys** to any mutation that might be retried.
- Next notebook: concurrency, geo search, caching, and rate limiting — with running code.